# 10 — DOE — measurements (Madina's CSV)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/10_doe_measurements.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/10_doe_measurements.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/10_doe_measurements.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Drop a measurement CSV in, get the same predicted-vs-measured comparison the dashboard's DOE tab produces — parity scatter, residuals, Q-Q, and a measured-vs-predicted effects tornado.

**Mirrors.** **DOE** tab → measurements upload + scatter + residuals + measured-tornado.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. The workflow

1. Generate a DOE design (any of notebooks 06–08, or the dashboard).
2. Run each row in the lab and record P9 (MPa), optionally P8 strain and P7 deformation, with one column per replicate.
3. Drop the file in and point `MEAS_PATH` at it.
4. Run the rest of this notebook — it produces the same comparison the DOE tab does (parity scatter, residual diagnostics, measured-vs-predicted tornado).

The CSV format and parser are identical to the dashboard's, so the numbers must match.


In [ ]:
from pathlib import Path
from cubespec import DEFAULT_CSP, full_factorial, main_effects
from cubespec.measurements import parse_measurements, write_template
import numpy as np, pandas as pd

# Step 1 — design (full 2-level on all 7 inputs = 128 runs).
design = full_factorial(DEFAULT_CSP, levels=2).reset_index(drop=True)
design['run'] = np.arange(1, len(design) + 1)
design = design[['run'] + [c for c in design.columns if c != 'run']]
design[['run', 'P0_rho', 'P1_dx', 'P4_Fx', 'P9_compressive_strength']].head()


## 2. Synthetic measurements (so the notebook runs in CI without a real CSV)

We add ±1.2 MPa Gaussian noise to the predicted P9 to simulate what a lab would record. **Replace this cell with `parse_measurements(MEAS_PATH)`** once you have a real file.


In [ ]:
rng = np.random.default_rng(42)
synthetic_path = Path('/tmp/madina_synthetic.csv')
with synthetic_path.open('w') as fh:
    fh.write('run,strength_mpa_1,strength_mpa_2,p8_strain,p7_def\n')
    for i in range(len(design)):
        true_p9 = float(design['P9_compressive_strength'].iloc[i])
        run_id = int(design['run'].iloc[i])
        m1 = true_p9 + rng.normal(0, 1.2)
        m2 = true_p9 + rng.normal(0, 1.2)
        fh.write(f'{run_id},{m1:.3f},{m2:.3f},,\n')
print('wrote', synthetic_path)

MEAS_PATH = synthetic_path  # <- replace with Path('/path/to/your.csv')
ms = parse_measurements(MEAS_PATH)
print('detected columns:', ms.columns_detected)
ms.to_dataframe().head()


## 3. Predicted vs measured — parity scatter


In [ ]:
joined = ms.align(design, output='P9_compressive_strength')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.5, 5.5))
lo = float(min(joined.predicted.min(), joined.measured.min())) - 1
hi = float(max(joined.predicted.max(), joined.measured.max())) + 1
ax.plot([lo, hi], [lo, hi], color='black', lw=0.8, ls='--', alpha=0.6, label='y = x')
ax.scatter(joined.predicted, joined.measured, s=44, color='#5b8def', alpha=0.85)
ax.set_xlabel('predicted P9 (MPa)'); ax.set_ylabel('measured P9 (MPa)')
ax.set_title('Predicted vs measured — Madina set'); ax.legend()
fig.tight_layout(); plt.show()


## 4. Residual diagnostics (RMSE, MAE, bias, R²)


In [ ]:
from cubespec import rmse, mae, r2, bias
yp, ym = joined.predicted.to_numpy(), joined.measured.to_numpy()
print(f'N runs compared : {len(yp)}')
print(f'RMSE            : {rmse(ym, yp):.3f} MPa')
print(f'MAE             : {mae(ym, yp):.3f} MPa')
print(f'bias            : {bias(ym, yp):+.3f} MPa  (positive = surrogate under-predicts)')
print(f'R²              : {r2(ym, yp):.3f}')


## 5. Q-Q plot of residuals


In [ ]:
from cubespec import qq_pairs
theo, obs = qq_pairs(joined.residual.to_numpy())
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(theo, obs, 'o', color='#e94f64', alpha=0.85)
lim = float(max(abs(theo).max(), abs(obs).max(), 1))
ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.6)
ax.set_xlabel('theoretical normal quantile')
ax.set_ylabel('observed residual quantile')
ax.set_title('Q-Q plot -- residuals against N(0, sigma_hat)')
fig.tight_layout(); plt.show()


## 6. Measured-effects vs predicted-effects tornado

Re-derive the main effects from the measured P9 column and compare them to the surrogate's predicted main effects.


In [ ]:
# Compare predicted main effects against a measured-effect estimate.
from cubespec.params import PARAM_KEYS
factors = ['P0_rho', 'P1_dx', 'P4_Fx']
pe = (main_effects(design)
      .query("output == 'P9_compressive_strength' and factor in @factors")
      .set_index('factor').effect)

# Manual measured main-effect = mean(measured | high coded) - mean(measured | low coded).
joined_design = joined.merge(design[['run'] + [f'{k}_coded' for k in factors]], on='run')
me_meas = {}
for f in factors:
    col = f'{f}_coded'
    hi = joined_design.loc[joined_design[col] > 0, 'measured'].mean()
    lo = joined_design.loc[joined_design[col] < 0, 'measured'].mean()
    me_meas[f] = hi - lo
me_meas = pd.Series(me_meas)
comp = pd.DataFrame({'predicted': pe, 'measured': me_meas}).sort_values('predicted', key=abs)
ax = comp.plot.barh(figsize=(7, 3.4), color=['#5b8def', '#e94f64'])
ax.set_xlabel('main effect on P9 (MPa)')
ax.set_title('Main effects — predicted vs measured (Madina set)')
plt.tight_layout(); plt.show()


## 7. One-paragraph summary you can paste into the report


In [ ]:
summary = (
    f'Compared {len(yp)} runs from {MEAS_PATH.name}. '
    f'RMSE = {rmse(ym, yp):.2f} MPa, R² = {r2(ym, yp):.3f}, '
    f'bias = {bias(ym, yp):+.2f} MPa. '
    f'The surrogate is calibrated against 500 synthetic runs at seed 1337; '
    f'expected RMSE for that fit is ≈ 1.0 MPa. '
    f'Your measurements {"agree with" if r2(ym, yp) > 0.85 else "deviate from"} '
    f'the surrogate within the calibrated tolerance.'
)
print(summary)


## 8. Template CSV

If you do not yet have data, write the blank template:


In [ ]:
blank = write_template('/tmp/madina_blank.csv', n_runs=8)
print('Wrote', blank); print(blank.read_text())


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
